# 01 — Ingest, validate, clean, enrich

Part 1 (26 marks). Run with **Restart Kernel → Run All**. Outputs are written to the **repo root** (hand-off files for Parts 2–3; `data/` keeps only the three originals): `rejected_rows.json`, `clinics_enriched.csv`, `appointments_clean.csv`, `load_data.sql`.

## Setup — imports & paths

In [ ]:
lazy import csv
lazy import json
lazy import requests
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent               # notebook lives in "Working with init Data/"; repo root is one up
DATA_DIR = ROOT / "data"
APPOINTMENTS_CSV = DATA_DIR / "appointments.csv"
CLINICS_CSV = DATA_DIR / "clinics.csv"

assert APPOINTMENTS_CSV.exists(), f"{APPOINTMENTS_CSV} not found - open the notebook from its own folder"

## Task 1.1 — Read the file the foundations way (csv module only)

### 1.1.1 Open with DictReader — header & row count

In [ ]:
with open(APPOINTMENTS_CSV, newline="") as f:
    reader = csv.DictReader(f)
    header = reader.fieldnames
    raw_rows = list(reader)

print("Header:", header)
print("Rows read:", len(raw_rows))

### 1.1.2 `parse_row(row)`

In [ ]:
INT_FIELDS = ("age", "fee_inr", "prior_no_shows")
FLOAT_FIELDS = ("distance_km",)


def parse_row(row: dict) -> dict:
    """Return a typed copy of one CSV row; raise ValueError if a numeric field won't convert."""
    parsed = dict(row)  # copy, so a rejected row keeps its raw strings
    for field in INT_FIELDS:
        try:
            parsed[field] = int(row[field])
        except ValueError:
            raise ValueError(f"Invalid integer for '{field}': {row[field]!r}") from None
    for field in FLOAT_FIELDS:
        try:
            parsed[field] = float(row[field])
        except ValueError:
            raise ValueError(f"Invalid float for '{field}': {row[field]!r}") from None
    return parsed

### 1.1.3 Accept / reject loop

In [ ]:
accepted, rejected = [], []
reasons = Counter()

for row in raw_rows:
    try:
        accepted.append(parse_row(row))
    except ValueError as err:
        rejected.append(row)
        reasons[str(err).split("'")[1]] += 1   # field name from the error message

print(f"Accepted: {len(accepted)}, Rejected: {len(rejected)}")
print("Rejected by field:", dict(reasons))

### 1.1.4 Save `rejected_rows.json`

In [ ]:
with open(ROOT / "rejected_rows.json", "w") as f:
    json.dump(rejected, f, indent=2)

print(f"Saved {len(rejected)} rows to rejected_rows.json")

A type check is not a validity check: rows with impossible ages (e.g. 250) are *accepted* here because they are valid integers. Validity is handled in Task 1.3.

## Task 1.2 — Inspect and clean, pass 1 (pandas, fresh from raw CSV)

### 1.2.0 Load & inspect — shape, info, nulls, duplicates

In [ ]:
df = pd.read_csv(APPOINTMENTS_CSV)

print("Shape:", df.shape)
df.info()
print("\nNulls per column:\n", df.isnull().sum(), sep="")
print("\nExact duplicate rows:", df.duplicated().sum())
print("booking_channel raw spellings:", df["booking_channel"].nunique())

### 1.2(a) Drop exact duplicates

In [ ]:
df = df.drop_duplicates()
print("Rows after (a) drop duplicates:", len(df))

### 1.2(b) Clean `booking_channel`

In [ ]:
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("Rows after (b):", len(df))
print("booking_channel distinct after cleaning:", df["booking_channel"].nunique(), sorted(df["booking_channel"].unique()))

### 1.2(c) Convert `appt_date` & `booked_on`

In [ ]:
had_appt_date = df["appt_date"].notna()
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])

unreadable = (had_appt_date & df["appt_date"].isna()).sum()
print("Rows after (c):", len(df))
print("Unparseable appt_date text -> NaT:", unreadable, "(plus 1 'N/A' pandas already read as missing)")
print("Missing appt_date total:", df["appt_date"].isna().sum(), "| missing status:", df["status"].isna().sum())

### 1.2(d) Drop rows missing `appt_date` / `status`

In [ ]:
df = df.dropna(subset=["appt_date", "status"]).reset_index(drop=True)
print("Rows after (d):", len(df))

## Task 1.3 — Clean, pass 2: validity, gaps, outliers

### 1.3(a) Invalid ages → missing

In [ ]:
AGE_MIN, AGE_MAX = 1, 100

invalid_age = (df["age"] < AGE_MIN) | (df["age"] > AGE_MAX)
print("Invalid ages:", invalid_age.sum(), "->", sorted(df.loc[invalid_age, "age"].tolist()))
df.loc[invalid_age, "age"] = np.nan
print("Missing ages now:", df["age"].isna().sum())

### 1.3(b) Fill `age` & `distance_km` with medians

In [ ]:
age_median = df["age"].median()
distance_median = df["distance_km"].median()

df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(distance_median)

print("Median age:", age_median, "| median distance_km:", distance_median)
print("Missing after fill -> age:", df["age"].isna().sum(), "distance_km:", df["distance_km"].isna().sum())

### 1.3(c) IQR caps on `distance_km` & `fee_inr`

In [ ]:
IQR_MULTIPLIER = 1.5

for col in ["distance_km", "fee_inr"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - IQR_MULTIPLIER * iqr, q3 + IQR_MULTIPLIER * iqr
    n_low, n_high = (df[col] < low).sum(), (df[col] > high).sum()
    df[col] = np.clip(df[col], low, high)
    print(f"{col}: Q1={q1}, Q3={q3}, IQR={iqr}, fences=({low}, {high}) -> capped {n_low + n_high} (low {n_low}, high {n_high})")

### 1.3(d) Round

In [ ]:
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)

print("Largest fee after capping:", df["fee_inr"].max())
print("distance_km range:", df["distance_km"].min(), "-", df["distance_km"].max())

## Task 1.4 — Features and a NumPy summary

### 1.4.0 `lead_days` & `no_show`

In [ ]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

print("lead_days range:", df["lead_days"].min(), "-", df["lead_days"].max())
print(f"No-shows: {df['no_show'].sum()} of {len(df)} ({df['no_show'].mean() * 100:.1f}%)")

### 1.4.1 `X` array

In [ ]:
FEATURES = ["age", "distance_km", "fee_inr", "lead_days"]
X = df[FEATURES].to_numpy()
print("X shape:", X.shape)

### 1.4(a) Column means

In [ ]:
col_means = X.mean(axis=0)
print(dict(zip(FEATURES, col_means.round(2).tolist())))

### 1.4(b) `lead_days > 14` mask

In [ ]:
lead_days = X[:, FEATURES.index("lead_days")]
mask = lead_days > 14
print(f"lead_days > 14: {mask.sum()} rows ({mask.mean() * 100:.1f}%)")

### 1.4(c) z-scores of `distance_km`

In [ ]:
x = X[:, FEATURES.index("distance_km")]
z = (x - np.mean(x)) / np.std(x)          # np.std = population std (ddof=0)
print("Rows with |z| > 2:", (np.abs(z) > 2).sum())

## Task 1.5 — Enrich the clinics from a live API

### 1.5.1 Load clinics

In [ ]:
clinics = pd.read_csv(CLINICS_CSV)
clinics

### 1.5.2 Open-Meteo call per clinic (timeout, retry once, None on failure)

In [ ]:
OPEN_METEO_URL = "https://api.open-meteo.com/v1/forecast"
TIMEOUT_S = 10
ATTEMPTS = 2   # first try + one retry


def fetch_temp_c(lat: float, lon: float) -> tuple[float | None, str | None]:
    """Return (temp_c, None) on success or (None, reason) on failure. Never invents a value."""
    params = {"latitude": lat, "longitude": lon, "hourly": "temperature_2m", "forecast_days": 1}
    reason = None
    for _ in range(ATTEMPTS):
        try:
            response = requests.get(OPEN_METEO_URL, params=params, timeout=TIMEOUT_S)
        except requests.exceptions.RequestException as err:
            reason = f"request failed: {err}"
            continue                                    # retry once
        if response.status_code != 200:
            return None, f"status {response.status_code}"
        data = response.json()
        if "hourly" not in data:
            return None, "reply has no 'hourly' field"
        return data["hourly"]["temperature_2m"][0], None
    return None, reason


temps = []
for clinic in clinics.itertuples():
    temp_c, reason = fetch_temp_c(clinic.latitude, clinic.longitude)
    if reason:
        print(f"{clinic.clinic_name}: temp_c = None ({reason})")
    temps.append(temp_c)

clinics["temp_c"] = temps
clinics

### 1.5.3 Save `clinics_enriched.csv`

In [ ]:
clinics.to_csv(ROOT / "clinics_enriched.csv", index=False)
print(f"Saved clinics_enriched.csv: {clinics.shape}, missing temp_c: {clinics['temp_c'].isna().sum()}")

## Task 1.6 — Export the clean table and write the SQL load script

### 1.6.1 Checks — unique `appointment_id`, no missing values

In [ ]:
EXPORT_COLUMNS = [
    "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date", "age", "gender",
    "distance_km", "fee_inr", "booking_channel", "visit_type", "reminder_sent",
    "prior_no_shows", "status", "lead_days", "no_show",
]

ids_unique = df["appointment_id"].is_unique
missing_total = df[EXPORT_COLUMNS].isnull().sum().sum()
print(f"appointment_id unique: {ids_unique} ({df['appointment_id'].nunique()} distinct)")
print("Missing values across the 16 columns:", missing_total)

assert ids_unique, "Duplicate appointment_id values"
assert missing_total == 0, "Missing values in export columns"

### 1.6.2 Sort, format dates, save `appointments_clean.csv` (16 columns)

In [ ]:
clean = df.sort_values("appointment_id")[EXPORT_COLUMNS].reset_index(drop=True)
for col in ["booked_on", "appt_date"]:
    clean[col] = clean[col].dt.strftime("%Y-%m-%d")

clean.to_csv(ROOT / "appointments_clean.csv", index=False)
print("Saved appointments_clean.csv:", clean.shape)

### 1.6.3 Write `load_data.sql` — clinics first, then appointments

In [ ]:
def sql_value(value) -> str:
    """Text in single quotes, numbers unquoted, missing as NULL."""
    if pd.isna(value):
        return "NULL"
    if isinstance(value, str):
        return f"'{value}'"
    return f"{value}"


def insert_statement(table: str, row) -> str:
    return f"INSERT INTO {table} VALUES ({','.join(sql_value(v) for v in row)});"


lines = []
for row in clinics.itertuples(index=False):          # clinics first: appointments.clinic_id is a foreign key
    lines.append(insert_statement("clinics", row))
for row in clean.itertuples(index=False):
    lines.append(insert_statement("appointments", row))

with open(ROOT / "load_data.sql", "w") as f:
    f.write("\n".join(lines) + "\n")

n_clinics = sum(l.startswith("INSERT INTO clinics") for l in lines)
print(f"load_data.sql: {n_clinics} + {len(lines) - n_clinics} = {len(lines)} INSERT statements")
print(lines[0])
print(lines[n_clinics])